# MyAI K3 — 100M base on real 10M data (GPU, ~5h)
Self-sufficient: stream 10M, 32k BPE --fast, pack-512, train MyAI-100M (109.5M params) x5000 AMP. ckpt/500 + eval/500. Resumable via --resume if the session interrupts.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1
!python scripts/stream_fineweb.py --max-tokens 10000000 --out data/raw/fineweb-10M.txt
!python scripts/prepare_tokenizer_corpus.py
!python scripts/train_tokenizer.py --corpus data/tokenizer_corpus/train.txt --vocab-size 32000 --out data/tokenizer_corpus/tokenizer-32k.json --fast
!test -f data/tokenizer_corpus/tokenizer-32k.json && echo TOKENIZER_OK

In [ ]:
%%writefile configs/data-pack-fw10m.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fw10m-32k
seq_len: 512
val_ratio: 0.01

In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-fw10m.yaml --rows-per-shard 2048
!test -f data/processed/fw10m-32k/manifest.json && echo PACK_OK

In [ ]:
!python scripts/train.py --model-config configs/myai-100m.yaml --data-dir data/processed/fw10m-32k --max-steps 5000 --batch-size 4 --accum 8 --amp --eval-every 500 --ckpt-every 500 --out-dir /kaggle/working/ckpts-100m --report /kaggle/working/train-100m.json

In [ ]:
!python scripts/eval_model.py --checkpoint /kaggle/working/ckpts-100m/final.pt --data-dir data/processed/fw10m-32k --report /kaggle/working/eval-100m.json
!ls -la /kaggle/working/ckpts-100m/ && cat /kaggle/working/train-100m.json